# Трениране на Невронна Мрежа (Версия 6) върху Преструктурирани Данни

Този notebook демонстрира процеса на трениране на модел V6, използвайки новите характеристики, създадени на базата на PDP анализа.

**Цели:**
1.  Зареждане на преструктурираните данни (`dataset/processed_v5/`).
2.  Дефиниране на архитектурата на модела (Dense слоеве, Dropout, BatchNormalization).
3.  Трениране с оптимизатор SGD и подходящи callbacks (EarlyStopping).
4.  Оценка на представянето (AUC, Accuracy, Confusion Matrix).

In [ ]:
# 1. Импортиране на библиотеки
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve
from sklearn.utils import class_weight
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Конфигурация за запазване
MODEL_SAVE_PATH = 'models/neural_network/models_versions/synthetic_model_v6.keras'
REPORT_DIR = 'models/neural_network/reports/report_v6'
os.makedirs(REPORT_DIR, exist_ok=True)

# 2. Зареждане на Данни
Тук зареждаме вече подготвените (скалирани, балансирани и кодирани) данни от папката `dataset/processed_v5/`.

In [ ]:
PROCESSED_DATA_DIR = 'dataset/processed_v5'

print("Зареждане на данни...")
X_train = pd.read_csv(f'{PROCESSED_DATA_DIR}/X_train.csv')
y_train = pd.read_csv(f'{PROCESSED_DATA_DIR}/y_train.csv').values.ravel()

X_val = pd.read_csv(f'{PROCESSED_DATA_DIR}/X_val.csv')
y_val = pd.read_csv(f'{PROCESSED_DATA_DIR}/y_val.csv').values.ravel()

print(f"Размер на тренировъчния сет: {X_train.shape}")
print(f"Размер на валидационния сет: {X_val.shape}")

# 3. Дефиниране на Модела
Изграждаме Sequential модел с няколко скрити слоя. Използваме `Dropout` и `BatchNormalization` за предотвратяване на преобучение (overfitting) и за по-стабилно обучение.

In [ ]:
def build_model(input_dim):
    model = Sequential([
        Input(shape=(input_dim,)),
        Dense(128, activation='relu'),
        BatchNormalization(),
        Dropout(0.4),
        Dense(64, activation='relu'),
        BatchNormalization(),
        Dropout(0.3),
        Dense(32, activation='relu'),
        BatchNormalization(),
        Dropout(0.2),
        Dense(1, activation='sigmoid')
    ])
    
    model.compile(
        optimizer=SGD(learning_rate=0.01, momentum=0.9),
        loss='binary_crossentropy',
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
    )
    return model

model = build_model(X_train.shape[1])
model.summary()

# 4. Трениране
Стартираме процеса на обучение. Използваме `ReduceLROnPlateau`, за да намалим скоростта на обучение, ако метриките спрат да се подобряват, и `EarlyStopping`, за да спрем процеса, ако няма подобрение твърде дълго.

In [ ]:
# Изчисляване на тегла на класовете (за всеки случай, въпреки SMOTE)
weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weights = dict(enumerate(weights))

callbacks = [
    ReduceLROnPlateau(monitor='val_auc', mode='max', factor=0.5, patience=5, min_lr=0.00001, verbose=1),
    EarlyStopping(monitor='val_auc', mode='max', patience=12, restore_best_weights=True)
]

print("Стартиране на тренировката...")
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=64,
    class_weight=class_weights,
    callbacks=callbacks,
    verbose=1
)

model.save(MODEL_SAVE_PATH)
print(f"Моделът е запазен в: {MODEL_SAVE_PATH}")

# 5. Оценка и Визуализация
Визуализираме историята на обучението (Loss и AUC) и матрицата на объркване, за да оценим колко добре се справя моделът.

In [ ]:
# История на обучението
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['auc'], label='Train AUC')
plt.plot(history.history['val_auc'], label='Val AUC')
plt.legend()
plt.title('AUC History')

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.legend()
plt.title('Loss History')
plt.show()

# Confusion Matrix
y_pred_prob = model.predict(X_val)
y_pred = (y_pred_prob > 0.5).astype(int)
cm = confusion_matrix(y_val, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

# Финални метрики
val_loss, val_acc, val_auc = model.evaluate(X_val, y_val)
print(f"Validation AUC: {val_auc:.4f}")
print(f"Validation Accuracy: {val_acc:.4f}")